In [7]:
import logging

import matplotlib.pyplot as plt
from numpy.random import RandomState

from sklearn import cluster, decomposition
from sklearn.preprocessing import MinMaxScaler
import pickle as pkl
import numpy as np
rng = RandomState(0)
from torch.utils.data import DataLoader, TensorDataset
# Display progress logs on stdout
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import torch
import torch.nn as nn
device='cpu'

In [2]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=64,kernel_size=5, padding='same')
        self.conv2=nn.Conv2d(in_channels=64,out_channels=128,kernel_size=3, padding='same')
        self.conv3=nn.Conv2d(in_channels=128,out_channels=256,kernel_size=5, padding='same')
        
        self.maxPooling2=nn.MaxPool2d(kernel_size=2)
        self.maxPooling4_0=nn.MaxPool2d(kernel_size=4)
        self.maxPooling4_1=nn.MaxPool2d(kernel_size=4)
#         self.adPooling=nn.AdaptiveAvgPool1d(256)
        
        self.fc1=nn.Linear(in_features=12544,out_features=128)
        self.fc2=nn.Linear(in_features=128,out_features=64)
        self.out=nn.Linear(in_features=64,out_features=2)

    def forward(self,x):
        x=self.conv1(x)
        x=self.maxPooling4_0(x)
        x=F.relu(x)
        
        x=self.conv2(x)
        x=self.maxPooling4_1(x)
        x=F.relu(x)
        
        x=self.conv3(x)
        x=self.maxPooling2(x)
        x=F.relu(x)
        
        x=F.dropout(x)
        x=x.view(1,x.size()[0],-1) #stretch to 1d data
        #x=self.adPooling(x).squeeze()
        
        x=self.fc1(x)
        x=F.relu(x)
        
        x=self.fc2(x)
        x=F.relu(x)
        
        x=self.out(x)
        
        return x[0]

In [28]:
batch_size = 64
split = 0

with open(f'./artifacts/split_{split}_coco_data_norm_test.pkl', 'rb') as f:
    [x_test_conf, y_test_conf, masks_test_conf, _] = pkl.load(f)

test_loader = DataLoader(TensorDataset(torch.tensor(x_test_conf.transpose(0,3,1,2)), torch.tensor(y_test_conf)), batch_size=batch_size, shuffle=False, num_workers=4)

In [29]:
with open(f'./artifacts/split_{split}_coco_data_norm_test.pkl', 'rb') as f:
    data = pkl.load(f)

In [9]:
def load_model(path, net=Net()):
    model = net
    
    model.load_state_dict(torch.load(path,map_location=device))
    model.eval()
    model.zero_grad()
    
    model.relu=nn.ReLU(inplace=False)
    return model

[tensor([[[[0.1451, 0.1373, 0.1490,  ..., 0.2196, 0.1961, 0.1765],
          [0.1451, 0.1490, 0.1373,  ..., 0.1804, 0.2353, 0.1765],
          [0.1490, 0.1529, 0.1451,  ..., 0.1686, 0.2078, 0.1882],
          ...,
          [0.1451, 0.1412, 0.1373,  ..., 0.1412, 0.1412, 0.1294],
          [0.1412, 0.1333, 0.1255,  ..., 0.1412, 0.1412, 0.1255],
          [0.1373, 0.1255, 0.1216,  ..., 0.1412, 0.1373, 0.1333]],

         [[0.4690, 0.4726, 0.4699,  ..., 0.4617, 0.4095, 0.4010],
          [0.4775, 0.4775, 0.4576,  ..., 0.3454, 0.4726, 0.4264],
          [0.5084, 0.5084, 0.5090,  ..., 0.3699, 0.4053, 0.1907],
          ...,
          [0.7246, 0.7246, 0.6991,  ..., 0.6265, 0.5543, 0.8578],
          [0.6678, 0.7408, 0.8420,  ..., 0.5384, 0.5967, 0.8916],
          [0.6326, 0.8318, 0.7246,  ..., 0.5672, 0.5525, 0.8374]],

         [[0.5137, 0.4784, 0.5176,  ..., 0.2118, 0.2745, 0.3686],
          [0.5176, 0.4706, 0.5529,  ..., 0.4078, 0.1647, 0.3804],
          [0.4510, 0.3255, 0.3804,  ..., 

SystemExit: 0

/home/clark01/mambaforge/envs/debugging/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [17]:
x[0].shape

torch.Size([64, 3, 224, 224])

In [22]:
from captum.attr import InputXGradient
import torch.nn.functional as F
from torchvision import models
DEVICE = 'cpu'

for batch in test_loader:
    # print(batch)
    break

model = load_model('./models/coco_conf_0_0.pt', net=Net()).eval().to(DEVICE)

sample = batch[0][0].reshape(1,3,224,224).to(torch.float32).to(DEVICE).reshape(1,3,224,224).to(torch.float32).to(DEVICE).reshape(1,3,224,224).to(torch.float32).to(DEVICE)
label = batch[1][0].to(torch.int16).to(DEVICE)

ig_att = InputXGradient(model).attribute(sample.to(DEVICE,dtype=torch.float), target=label).squeeze()
# ig_att = InputXGradient(model).attribute(w_im_rot.to(DEVICE,dtype=torch.float), target=w_target).squeeze()
# ig_att = InputXGradient(model).attribute(torch.tensor(noise).to(DEVICE,dtype=torch.float), target=w_target).squeeze()

/tmp/ipykernel_1414307/3380686188.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path,map_location=device))
